In [ ]:
#@title ⚙️ שלב 0 — הגדרות פרויקט ובחירת מעבדה  { run: "auto" }

# ── פרטי הפרויקט ──────────────────────────────────────────────
project_name  = "שם הפרויקט"  #@param {type:"string"}
client_name   = "שם המזמין"   #@param {type:"string"}
output_file   = "lab_report"  #@param {type:"string"}

# ── בחירת מעבדה ────────────────────────────────────────────────
lab = "KTE" #@param ["Alchem", "KTE", "מכון הנפט", "בקטוכם"]

# ── קטגוריית ניתוח (auto = זיהוי אוטומטי לפי שם הקובץ) ────────
analysis_category = "auto" #@param ["auto", "soil_gas", "soil", "groundwater", "pfas", "pr"]

# ── ערכי סף — אחד או יותר ────────────────────────────────────
include_VSL_soil          = True   #@param {type:"boolean"}
include_TIER1_indoor_res  = False  #@param {type:"boolean"}
include_TIER1_outdoor_res = False  #@param {type:"boolean"}
include_TIER1_indoor_ind  = False  #@param {type:"boolean"}
include_TIER1_outdoor_ind = False  #@param {type:"boolean"}
include_GW                = True   #@param {type:"boolean"}
include_PFAS_VSL          = True   #@param {type:"boolean"}
include_PFAS_tier1_res    = False  #@param {type:"boolean"}
include_PFAS_tier1_ind    = False  #@param {type:"boolean"}

# ── בנה רשימת ערכי סף נבחרים ──────────────────────────────────
selected_thresholds = []
if include_VSL_soil:          selected_thresholds.append('VSL_SOIL')
if include_TIER1_indoor_res:  selected_thresholds.append('TIER1_INDOOR_RES')
if include_TIER1_outdoor_res: selected_thresholds.append('TIER1_OUTDOOR_RES')
if include_TIER1_indoor_ind:  selected_thresholds.append('TIER1_INDOOR_IND')
if include_TIER1_outdoor_ind: selected_thresholds.append('TIER1_OUTDOOR_IND')
if include_GW:                selected_thresholds.append('GW')
if include_PFAS_VSL:          selected_thresholds.append('PFAS_VSL')
if include_PFAS_tier1_res:    selected_thresholds.append('PFAS_TIER1_RES')
if include_PFAS_tier1_ind:    selected_thresholds.append('PFAS_TIER1_IND')

print(f'✅ פרמטרים:')
print(f'   פרויקט   : {project_name}')
print(f'   מזמין    : {client_name}')
print(f'   מעבדה    : {lab}')
print(f'   קטגוריה  : {analysis_category}')
print(f'   ערכי סף  : {selected_thresholds}')
print(f'   קובץ פלט : {output_file}.xlsx')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — חיבור Google Drive + הגדרת נתיבים
# ═══════════════════════════════════════════════════════════════════
from google.colab import drive
import sys, os, subprocess

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/laboratory_results_analsys/soil_lab_tool'
THRESH_DIR   = os.path.join(PROJECT_ROOT, 'thresholds')
LAB_DIR      = os.path.join(os.path.dirname(PROJECT_ROOT), 'Laboratory_results')

if not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError(
        f'❌ תיקיית הפרויקט לא נמצאה: {PROJECT_ROOT}\n'
        'ודא שהתיקייה soil_lab_tool נמצאת ב-Drive'
    )

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

subprocess.run(['pip', 'install', 'openpyxl', 'pandas', '-q'], check=True)

print(f'✅ Drive מחובר')
print(f'📁 PROJECT_ROOT  = {PROJECT_ROOT}')
print(f'📁 THRESHOLDS    = {THRESH_DIR}')
print(f'📁 LAB_RESULTS   = {LAB_DIR}')
print()
print('📂 מבנה parsers:')
parsers_dir = os.path.join(PROJECT_ROOT, 'parsers')
for root, dirs, files in os.walk(parsers_dir):
    dirs[:] = [d for d in sorted(dirs) if d != '__pycache__']
    level  = root.replace(parsers_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(f for f in files if f.endswith('.py')):
        print(f'  {indent}{f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — טעינת מודולים + אתחול ThresholdManager
# ═══════════════════════════════════════════════════════════════════
import importlib, sys

# Force-reload all custom modules
for mod in list(sys.modules.keys()):
    if mod.startswith(('core', 'parsers')):
        del sys.modules[mod]

print('📦 טוען מודולים...')
modules_to_check = [
    'core.cas_lookup',
    'core.lab_value_parser',
    'core.threshold_manager',
    'core.excel_output',
    'parsers',
    'parsers.soil_gas.alchem',
    'parsers.soil.alchem',
    'parsers.soil.kte',
    'parsers.soil.kte_pr',
    'parsers.soil.machon_haneft',
    'parsers.groundwater.kte',
    'parsers.groundwater.bactochem',
    'parsers.pfas.kte',
]
all_ok = True
for mod_path in modules_to_check:
    try:
        importlib.import_module(mod_path)
        print(f'  ✅  {mod_path}')
    except Exception as e:
        print(f'  ❌  {mod_path} → {e}')
        all_ok = False

if not all_ok:
    raise ImportError('תקן שגיאות imports לפני המשך')

# ── ThresholdManager ─────────────────────────────────────────────
from core.threshold_manager import ThresholdManager

MAIN_THRESH_FILE = os.path.join(THRESH_DIR, 'soil_vsl_tier1_v7_2024.xlsx')
VSL_FULL_FILE    = os.path.join(THRESH_DIR, 'soil_vsl_v7_full.xlsx')   # 800+ compounds
PFAS_THRESH_FILE = os.path.join(os.path.dirname(PROJECT_ROOT),
                                 'Laboratory_results',
                                 'נספח לטבלת ערכי סף - PFAS.xlsx')

pfas_path     = PFAS_THRESH_FILE if os.path.exists(PFAS_THRESH_FILE) else None
vsl_full_path = VSL_FULL_FILE    if os.path.exists(VSL_FULL_FILE)    else None

tm = ThresholdManager(MAIN_THRESH_FILE, pfas_path=pfas_path, vsl_full_path=vsl_full_path)

benzene_vsl  = tm.get_threshold('71-43-2', 'VSL_SOIL')
benzene_res  = tm.get_threshold('71-43-2', 'TIER1_INDOOR_RES')
benzene_gw   = tm.get_threshold('71-43-2', 'GW')
lead_vsl     = tm.get_threshold('7439-92-1', 'VSL_SOIL')
tph_dro_vsl  = tm.get_threshold('C10-C40',  'VSL_SOIL')

print(f'\n📋 ThresholdManager:')
print(f'   קובץ VSL מלא : {"✅ נטען (מתכות+TPH)" if vsl_full_path else "⏳ לא נמצא"}')
print(f'   קובץ PFAS    : {"✅ נטען" if pfas_path else "⏳ לא נמצא"}')
print(f'   בנזן VSL     : {benzene_vsl} mg/kg')
print(f'   בנזן Indoor  : {benzene_res} µg/m³')
print(f'   בנזן GW      : {benzene_gw} mg/L')
print(f'   עופרת VSL    : {lead_vsl} mg/kg')
print(f'   DRO+ORO VSL  : {tph_dro_vsl} mg/kg')
print(f'\n✅ אתחול הושלם')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — העלאת קובץ + פרסינג
# ═══════════════════════════════════════════════════════════════════
from google.colab import files as colab_files
from parsers import get_parser, auto_detect_category
import io, collections

print('📤 העלה קובץ דוח מעבדה (xlsx / xls / csv)...')
uploaded = colab_files.upload()

if not uploaded:
    raise ValueError('❌ לא הועלה אף קובץ')

filename   = list(uploaded.keys())[0]
file_bytes = uploaded[filename]
print(f'\n📄 קובץ: {filename}  ({len(file_bytes):,} bytes)')

# ── בחירת קטגוריה ────────────────────────────────────────────────
if analysis_category == 'auto':
    category = auto_detect_category(filename, file_bytes)
    print(f'🔍 זיהוי אוטומטי: category = "{category}"')
else:
    category = analysis_category
    print(f'📌 קטגוריה ידנית: "{category}"')

# ── Parser ───────────────────────────────────────────────────────
try:
    parser = get_parser(lab, category)
    print(f'🔬 Parser: {type(parser).__name__}')
except KeyError as e:
    print(f'⚠️  {e}')
    print('🔄 מנסה soil כברירת מחדל...')
    parser = get_parser(lab, 'soil')

# ── פרסינג ───────────────────────────────────────────────────────
file_obj = file_bytes if filename.endswith('.csv') else io.BytesIO(file_bytes)
records  = parser.parse(file_obj)

if not records:
    raise ValueError('❌ לא נמצאו רשומות — בדוק פורמט הקובץ ובחירת מעבדה')

# ── סטטיסטיקות ───────────────────────────────────────────────────
by_type = collections.Counter(r.get('analysis_type', '?') for r in records)
samples  = sorted(set(r['sample_id'] for r in records))
detected = [r for r in records if r.get('flag') != 'ND' and r.get('value') is not None]

print(f'\n📊 תוצאות פרסינג:')
print(f'   סה"כ רשומות : {len(records)}')
print(f'   ערכים מזוהים: {len(detected)}')
print(f'   דגימות      : {len(samples)}')
print(f'   לפי סוג:')
for atype, cnt in by_type.most_common():
    print(f'     {atype:20s} → {cnt} רשומות')
print(f'\n   שמות דגימות: {samples[:10]}{"..." if len(samples)>10 else ""}')

print('\n--- 5 רשומות ראשונות ---')
for r in records[:5]:
    flag  = r.get('flag', '')
    val   = r.get('value')
    vdisp = f'{val:.3g}' if isinstance(val, float) else str(val)
    print(f'  [{r["analysis_type"]}] {r["sample_id"]:15s}  '
          f'{r["compound"]:25s}  {vdisp:>10} {r.get("unit","")}  flag={flag}')

print('\n✅ פרסינג הושלם!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — בניית Excel פלט + הורדה
# ═══════════════════════════════════════════════════════════════════
from google.colab import files as colab_files
from core.excel_output import LabReportExcel
import os
from datetime import date

OUTPUT_PATH  = f'/content/{output_file}.xlsx'
TODAY        = date.today().strftime('%d.%m.%Y')

# Filter selected_thresholds to only the ones relevant to records in this file
# (avoids empty threshold columns for analysis types that don't use them)
thresh_override = selected_thresholds if selected_thresholds else None

print(f'🔄 בונה Excel...')
print(f'   פרויקט  : {project_name}')
print(f'   מזמין   : {client_name}')
print(f'   תאריך   : {TODAY}')
print(f'   ערכי סף : {thresh_override}')
print(f'   פלט     : {OUTPUT_PATH}')

builder = LabReportExcel(
    records             = records,
    threshold_manager   = tm,
    output_path         = OUTPUT_PATH,
    project_name        = project_name,
    client              = client_name,
    report_date         = TODAY,
    selected_thresholds = thresh_override,
)
builder.build()

size = os.path.getsize(OUTPUT_PATH)
print(f'\n✅ Excel נוצר: {output_file}.xlsx  ({size:,} bytes)')

# הצג גיליונות שנוצרו
import openpyxl
wb = openpyxl.load_workbook(OUTPUT_PATH, read_only=True)
print(f'📋 גיליונות: {wb.sheetnames}')
wb.close()

print('\n📥 מוריד...')
colab_files.download(OUTPUT_PATH)
print('✅ הכל הושלם!')